
# DQ Report for Team 1 Digi (v5)

Notebook vytvori a naplni tabulku `dq_report_v5` ve schematu `vse_banka.digi_prodej`.
Kontroluje prechody `bronze -> silver -> gold`, kontraktove gold produkty i logiku external customer profile enrichment.


In [0]:
from pyspark.sql import functions as F
import pandas as pd

SCHEMA = 'vse_banka.digi_prodej'
TEAM_ID = 'team1-digi'

BRONZE_PRODUCT = 'bronze_layer_quality'
SILVER_PRODUCT = 'silver_layer_quality'
ACQ_PRODUCT = 'strategy_marketing_performance_acquisition_aggregation'
PROP_PRODUCT = 'retail_sales_vehicle_insurance_propensity_state'
PROFILE_PRODUCT = 'external_customer_profile_enrichment'


In [0]:
spark.sql(f'''CREATE SCHEMA IF NOT EXISTS {SCHEMA}''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {SCHEMA}.dq_report_v5 
(
  report_date DATE,
  team_id STRING,
  data_product STRING,
  dq_dimension STRING,
  check_name STRING,
  total_records INT,
  passed_records INT,
  dq_score FLOAT
)
''')


DataFrame[]

In [0]:
report_date = spark.sql('SELECT CURRENT_DATE() AS d').collect()[0]['d']
checks = []

def add_check(data_product, dq_dimension, check_name, total_records, passed_records):
    total_records = int(total_records)
    passed_records = int(passed_records)
    dq_score = float(passed_records / total_records) if total_records else 0.0
    checks.append({
        'report_date': report_date,
        'team_id': TEAM_ID,
        'data_product': data_product,
        'dq_dimension': dq_dimension,
        'check_name': check_name,
        'total_records': total_records,
        'passed_records': passed_records,
        'dq_score': dq_score,
    })

def table_count(table_name):
    return spark.table(table_name).count()

def count_condition(table_name, condition):
    return spark.table(table_name).where(condition).count()

def distinct_count_expr(table_name, expr):
    row = spark.table(table_name).selectExpr(f'COUNT(DISTINCT {expr}) AS cnt').collect()[0]
    return int(row['cnt'])


In [0]:
# Bronze layer checks
bronze_web = f'{SCHEMA}.bronze_web_sessions'
total = table_count(bronze_web)
passed = count_condition(bronze_web, 'session_id IS NOT NULL AND user_id IS NOT NULL AND event_time IS NOT NULL AND source IS NOT NULL AND campaign IS NOT NULL')
add_check(BRONZE_PRODUCT, 'Completeness', 'bronze_web_sessions_required_fields_not_null', total, passed)

bronze_crm_cols = set(spark.table(f'{SCHEMA}.bronze_crm_clients').columns)
add_check(BRONZE_PRODUCT, 'Validity', 'bronze_crm_clients_region_column_absent', 1, 0 if 'region' in bronze_crm_cols else 1)
add_check(BRONZE_PRODUCT, 'Validity', 'bronze_crm_clients_age_band_column_absent', 1, 0 if 'age_band' in bronze_crm_cols else 1)

bronze_plan_cols = set(spark.table(f'{SCHEMA}.bronze_plan_digital_clients').columns)
add_check(BRONZE_PRODUCT, 'Validity', 'bronze_plan_digital_clients_region_column_absent', 1, 0 if 'region' in bronze_plan_cols else 1)
add_check(BRONZE_PRODUCT, 'Validity', 'bronze_plan_digital_clients_age_band_column_absent', 1, 0 if 'age_band' in bronze_plan_cols else 1)

total = table_count(f'{SCHEMA}.bronze_crm_clients') + table_count(f'{SCHEMA}.bronze_plan_digital_clients')
passed = count_condition(f'{SCHEMA}.bronze_crm_clients', "dq_flag IN ('PASS','WARN','FAIL')") + count_condition(f'{SCHEMA}.bronze_plan_digital_clients', "dq_flag IN ('PASS','WARN','FAIL')")
add_check(BRONZE_PRODUCT, 'Validity', 'bronze_clients_dq_flag_allowed_values', total, passed)


In [0]:
# Silver layer checks
silver_profile = f'{SCHEMA}.silver_customer_profile_enriched'
total = table_count(silver_profile)
passed = count_condition(silver_profile, 'client_id IS NOT NULL AND client_scope IS NOT NULL AND birth_date IS NOT NULL AND age_years IS NOT NULL AND age_band IS NOT NULL AND region IS NOT NULL')
add_check(PROFILE_PRODUCT, 'Completeness', 'silver_customer_profile_enriched_required_fields_not_null', total, passed)

total = table_count(f'{SCHEMA}.silver_customer_profile_link')
passed = distinct_count_expr(f'{SCHEMA}.silver_customer_profile_link', "concat_ws('||', client_id, client_scope)")
add_check(PROFILE_PRODUCT, 'Uniqueness', 'silver_customer_profile_link_client_scope_unique', total, passed)

total = table_count(silver_profile)
passed = count_condition(silver_profile, "age_band IN ('18-24','25-34','35-44','45-54','55+')")
add_check(PROFILE_PRODUCT, 'Validity', 'silver_customer_profile_enriched_age_band_allowed_values', total, passed)

silver_crm = f'{SCHEMA}.silver_crm_clients'
total = table_count(silver_crm)
passed = count_condition(silver_crm, 'birth_date IS NOT NULL AND age_band IS NOT NULL AND region IS NOT NULL')
add_check(SILVER_PRODUCT, 'Consistency', 'silver_crm_clients_profile_join_coverage', total, passed)

silver_plan = f'{SCHEMA}.silver_plan_digital_clients'
total = table_count(silver_plan)
passed = count_condition(silver_plan, 'birth_date IS NOT NULL AND age_band IS NOT NULL AND region IS NOT NULL')
add_check(SILVER_PRODUCT, 'Consistency', 'silver_plan_digital_clients_profile_join_coverage', total, passed)

total = table_count(silver_crm)
passed = count_condition(silver_crm, "DATE(created_at) <= DATE('2025-12-31')")
add_check(SILVER_PRODUCT, 'Timeliness', 'silver_crm_clients_created_at_not_in_future', total, passed)

total = table_count(silver_plan)
passed = count_condition(silver_plan, 'YEAR(created_at) = 2026')
add_check(SILVER_PRODUCT, 'Validity', 'silver_plan_digital_clients_created_at_in_2026', total, passed)


In [0]:
# Gold layer checks - acquisition
gold_cpa = f'{SCHEMA}.gold_mart_cpa_daily'
total = table_count(gold_cpa)
passed = count_condition(gold_cpa, 'record_id IS NOT NULL AND business_date IS NOT NULL AND region IS NOT NULL AND date IS NOT NULL AND campaign IS NOT NULL AND platform IS NOT NULL AND clients IS NOT NULL AND total_cost_czk IS NOT NULL')
add_check(ACQ_PRODUCT, 'Completeness', 'gold_mart_cpa_daily_required_fields_not_null', total, passed)

total = table_count(gold_cpa)
passed = distinct_count_expr(gold_cpa, "concat_ws('||', cast(date as string), campaign, platform)")
add_check(ACQ_PRODUCT, 'Uniqueness', 'gold_mart_cpa_daily_date_campaign_platform_unique', total, passed)

total = table_count(gold_cpa)
passed = count_condition(gold_cpa, "region IN ('CZ','SK')")
add_check(ACQ_PRODUCT, 'Validity', 'gold_mart_cpa_daily_region_allowed_values', total, passed)

total = table_count(gold_cpa)
passed = count_condition(gold_cpa, '(clients = 0 AND cpa_czk IS NULL) OR (clients > 0 AND ABS(cpa_czk - (total_cost_czk / clients)) < 0.05)')
add_check(ACQ_PRODUCT, 'Consistency', 'gold_mart_cpa_daily_formula_consistency', total, passed)

total = table_count(gold_cpa)
passed = count_condition(gold_cpa, 'YEAR(date) = 2026')
add_check(ACQ_PRODUCT, 'Timeliness', 'gold_mart_cpa_daily_plan_year_2026', total, passed)


In [0]:
# Gold layer checks - propensity
gold_state = f'{SCHEMA}.gold_state_vehicle_insurance_propensity'
total = table_count(gold_state)
passed = count_condition(gold_state, 'record_id IS NOT NULL AND business_date IS NOT NULL AND region IS NOT NULL AND client_id IS NOT NULL AND snapshot_date IS NOT NULL AND propensity_score IS NOT NULL AND segment IS NOT NULL AND age_band IS NOT NULL')
add_check(PROP_PRODUCT, 'Completeness', 'gold_state_vehicle_propensity_required_fields_not_null', total, passed)

total = table_count(gold_state)
passed = count_condition(gold_state, 'propensity_score BETWEEN 0 AND 1')
add_check(PROP_PRODUCT, 'Validity', 'gold_state_vehicle_propensity_score_in_range', total, passed)

total = table_count(gold_state)
passed = count_condition(gold_state, "segment IN ('LOW','MEDIUM','HIGH')")
add_check(PROP_PRODUCT, 'Validity', 'gold_state_vehicle_propensity_segment_allowed_values', total, passed)

total = table_count(gold_state)
passed = distinct_count_expr(gold_state, "concat_ws('||', client_id, cast(snapshot_date as string))")
add_check(PROP_PRODUCT, 'Uniqueness', 'gold_state_vehicle_propensity_client_snapshot_unique', total, passed)

total = table_count(gold_state)
passed = count_condition(gold_state, "snapshot_date = DATE('2025-12-31')")
add_check(PROP_PRODUCT, 'Timeliness', 'gold_state_vehicle_propensity_snapshot_date_expected', total, passed)

silver_crm_count = table_count(f'{SCHEMA}.silver_crm_clients')
gold_state_count = table_count(gold_state)
add_check(PROP_PRODUCT, 'Consistency', 'gold_state_vehicle_propensity_same_count_as_silver_crm_clients', silver_crm_count, silver_crm_count if gold_state_count == silver_crm_count else 0)


In [0]:
dq_df = pd.DataFrame(checks)
#spark.createDataFrame(dq_df).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{SCHEMA}.dq_report_v5')
#display(spark.table(f'{SCHEMA}.dq_report_v5').orderBy('data_product', 'dq_dimension', 'check_name'))
spark.createDataFrame(dq_df).write.mode('append').saveAsTable(f'{SCHEMA}.dq_report_v5')
display(spark.table(f'{SCHEMA}.dq_report_v5').orderBy(F.col('report_date').desc(), F.col('data_product'), F.col('dq_dimension'), F.col('check_name')))

report_date,team_id,data_product,dq_dimension,check_name,total_records,passed_records,dq_score
2026-04-19,team1-digi,bronze_layer_quality,Completeness,bronze_web_sessions_required_fields_not_null,268076,268076,1.0
2026-04-19,team1-digi,bronze_layer_quality,Validity,bronze_clients_dq_flag_allowed_values,35000,35000,1.0
2026-04-19,team1-digi,bronze_layer_quality,Validity,bronze_crm_clients_age_band_column_absent,1,1,1.0
2026-04-19,team1-digi,bronze_layer_quality,Validity,bronze_crm_clients_region_column_absent,1,1,1.0
2026-04-19,team1-digi,bronze_layer_quality,Validity,bronze_plan_digital_clients_age_band_column_absent,1,1,1.0
2026-04-19,team1-digi,bronze_layer_quality,Validity,bronze_plan_digital_clients_region_column_absent,1,1,1.0
2026-04-19,team1-digi,external_customer_profile_enrichment,Completeness,silver_customer_profile_enriched_required_fields_not_null,35000,35000,1.0
2026-04-19,team1-digi,external_customer_profile_enrichment,Uniqueness,silver_customer_profile_link_client_scope_unique,35000,35000,1.0
2026-04-19,team1-digi,external_customer_profile_enrichment,Validity,silver_customer_profile_enriched_age_band_allowed_values,35000,35000,1.0
2026-04-19,team1-digi,retail_sales_vehicle_insurance_propensity_state,Completeness,gold_state_vehicle_propensity_required_fields_not_null,10000,10000,1.0
